# 1. Initializations

## 1.1 General CPU/GPU Checks (NVIDIA cards)

In [ ]:
### global
import logging
import os
import shutil
from smartcheck.logger_config import setup_logger

setup_logger(logging.INFO)
print(f'Path [{os.environ["PATH"]}]')

# Test tensorflow GPU config
import tensorflow as tf
print("✅ TF GPU:", tf.config.list_physical_devices("GPU"))

## 1.2 General imports

In [ ]:
# Pour la manipulation de tableaux et Dataframes
import numpy as np
from timeit import default_timer as timer

# Pour la visualisation des performances
import matplotlib.pyplot as plt
%matplotlib inline

# Pour construire un réseau de neurone
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Conv2D
from tensorflow.keras.models import Model
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import Callback, ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

# Pour la transformation sur les images
from tensorflow.keras.layers import Rescaling
from tensorflow.keras.layers import Resizing
from tensorflow.keras.layers import RandomFlip
# do not work correctly : alternative : create own CustomRandom classes
# from tensorflow.keras.layers import RandomZoom
# from tensorflow.keras.layers import RandomRotation
# from tensorflow.keras.layers import RandomBrightness
# from tensorflow.keras.layers import RandomContrast
# from tensorflow.keras.layers import RandomTranslation 
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input, VGG16

# Utilitaire d'importation des dataset d'images
from keras.utils import image_dataset_from_directory

# Interpretation des modèles
import shap


## 1.3 Custom Fallback for broken layers of image randomization in tensorflow 2.10

In [ ]:
class CustomRandomRotation(tf.keras.layers.Layer):
    def __init__(self, factor=0.2, fill_mode="REFLECT", **kwargs):
        super().__init__(**kwargs)
        self.factor = factor
        self.fill_mode = fill_mode.upper()

    def get_config(self):
        config = super().get_config()
        config.update({"factor": self.factor, "fill_mode": self.fill_mode})
        return config

    def call(self, images, training=True):
        if not training:
            return images

        angle_max = self.factor * 3.14159265  # radians
        batch_size = tf.shape(images)[0]

        angles = tf.random.uniform(
            shape=[batch_size],
            minval=-angle_max,
            maxval=angle_max
        )

        return self._rotate(images, angles)

    def _rotate(self, images, angles):
        transform_matrices = self._get_rotation_matrices(angles)
        image_dims = tf.shape(images)[1:3]
        return tf.raw_ops.ImageProjectiveTransformV3(
            images=images,
            transforms=transform_matrices,
            output_shape=image_dims,
            interpolation="BILINEAR",
            fill_mode=self.fill_mode,
            fill_value=0.0
        )

    def _get_rotation_matrices(self, angles):
        cos_a = tf.math.cos(angles)
        sin_a = tf.math.sin(angles)
        zero = tf.zeros_like(cos_a)

        transforms = tf.stack([
            cos_a, -sin_a, zero,
            sin_a,  cos_a, zero,
        ], axis=1)

        return tf.pad(transforms, [[0, 0], [0, 2]], constant_values=0.0)

    def compute_output_shape(self, input_shape):
        return input_shape   


class CustomRandomZoom(tf.keras.layers.Layer):
    def __init__(self, zoom_range=0.2, **kwargs):
        super().__init__(**kwargs)
        self.zoom_range = zoom_range

    def call(self, inputs, training=True):
        if not training:
            return inputs
        # Compute random crop box
        batch_size = tf.shape(inputs)[0]
        height = tf.shape(inputs)[1]
        width = tf.shape(inputs)[2]
        zoom = tf.random.uniform([], 1.0 - self.zoom_range, 1.0 + self.zoom_range)
        new_height = tf.cast(tf.cast(height, tf.float32) / zoom, tf.int32)
        new_width = tf.cast(tf.cast(width, tf.float32) / zoom, tf.int32)
        cropped = tf.image.resize_with_crop_or_pad(inputs, new_height, new_width)
        resized = tf.image.resize(cropped, [height, width])
        return resized

    def get_config(self):
        config = super().get_config()
        config.update({
            "zoom_range": self.zoom_range
        })
        return config
    
    def compute_output_shape(self, input_shape):
        return input_shape


class CustomRandomBrightness(tf.keras.layers.Layer):
    def __init__(self, max_delta=0.2, **kwargs):
        super().__init__(**kwargs)
        self.max_delta = max_delta

    def call(self, inputs, training=True):
        if not training:
            return inputs
        return tf.image.random_brightness(inputs, self.max_delta)

    def get_config(self):
        config = super().get_config()
        config.update({
            "max_delta": self.max_delta
        })
        return config
        
    def compute_output_shape(self, input_shape):
        return input_shape


class CustomRandomContrast(tf.keras.layers.Layer):
    def __init__(self, lower=0.8, upper=1.2, **kwargs):
        super().__init__(**kwargs)
        self.lower = lower
        self.upper = upper

    def call(self, inputs, training=True):
        if not training:
            return inputs
        return tf.image.random_contrast(inputs, self.lower, self.upper)

    def get_config(self):
        config = super().get_config()
        config.update({
            "lower": self.lower,
            "upper": self.upper
        })
        return config
    def compute_output_shape(self, input_shape):
        return input_shape


class CustomRandomTranslation(tf.keras.layers.Layer):
    def __init__(self, height_factor, width_factor, **kwargs):
        super().__init__(**kwargs)
        self.height_factor = self._normalize_factor(height_factor)
        self.width_factor = self._normalize_factor(width_factor)

    def _normalize_factor(self, factor):
        if isinstance(factor, (tuple, list)):
            return factor
        return (-abs(factor), abs(factor))

    def call(self, inputs, training=True):
        if not training:
            return inputs

        inputs = tf.cast(inputs, tf.float32)
        batch_size = tf.shape(inputs)[0]
        img_height = tf.cast(tf.shape(inputs)[1], tf.float32)
        img_width = tf.cast(tf.shape(inputs)[2], tf.float32)

        translations_y = tf.random.uniform(
            [batch_size],
            minval=self.height_factor[0],
            maxval=self.height_factor[1]
        ) * img_height

        translations_x = tf.random.uniform(
            [batch_size],
            minval=self.width_factor[0],
            maxval=self.width_factor[1]
        ) * img_width

        ones = tf.ones_like(translations_x)
        zeros = tf.zeros_like(translations_x)
        c0 = tf.zeros_like(translations_x)
        c1 = tf.zeros_like(translations_x)

        # Matrices 3x3 aplaties → 8 coefficients
        transforms = tf.stack([
            ones, zeros, -translations_x,
            zeros, ones, -translations_y,
            c0, c1
        ], axis=1)

        outputs = tf.raw_ops.ImageProjectiveTransformV3(
            images=inputs,
            transforms=transforms,
            output_shape=tf.shape(inputs)[1:3],
            interpolation='BILINEAR',
            fill_mode='REFLECT',
            fill_value=0.0
        )
        return outputs

    def get_config(self):
        config = super().get_config()
        config.update({
            "height_factor": self.height_factor,
            "width_factor": self.width_factor
        })
        return config

    def compute_output_shape(self, input_shape):
        return input_shape

# 2. Loading and Data Enrichment

In [ ]:
def classer_images_par_age(
    repertoire_images,
    age_min=None,
    age_max=None
):
    """
    Organise les images dans des sous-dossiers en fonction de l'âge extrait du nom de fichier.
    - Peut filtrer une plage d'âges : age_min à age_max
    - Réindexe les âges sélectionnés en commençant à 0
    """

    ages_valides = set()

    # Étape 1 — Parcourir tous les fichiers et collecter les âges valides
    fichiers_eligibles = []
    for racine, _, fichiers in os.walk(repertoire_images):
        for fichier in fichiers:
            if fichier.lower().endswith(".jpg"):
                try:
                    age = int(fichier.split("_")[0])
                    if ((age_min is None or age >= age_min) and
                        (age_max is None or age <= age_max)):
                        fichiers_eligibles.append((fichier, age, racine))
                        ages_valides.add(age)
                except Exception as e:
                    print(f"⚠️ Ignoré : {fichier} (erreur : {e})")

    # Étape 2 — Réindexer les âges valides
    ages_valides = sorted(ages_valides)
    mapping_ages = {age: idx for idx, age in enumerate(ages_valides)}
    print(f"🎯 Mapping des âges : {mapping_ages}")

    # Étape 3 — Déplacer les fichiers vers les bons dossiers (réindexés)
    for fichier, age, racine in fichiers_eligibles:
        nouvelle_classe = str(mapping_ages[age])
        dest_dir = os.path.join(repertoire_images, nouvelle_classe)
        os.makedirs(dest_dir, exist_ok=True)

        chemin_source = os.path.join(racine, fichier)
        chemin_destination = os.path.join(dest_dir, fichier)

        try:
            shutil.move(chemin_source, chemin_destination)
        except Exception as e:
            print(f"❌ Erreur déplacement {fichier} : {e}")

    print("✅ Organisation terminée.")

In [ ]:
# dataset provenant de https://susanqq.github.io/UTKFace/ et décompressé localement avec application d'une reconstruction
# des sous-répertoires par classe avec la fonction utilitaire classer_images_par_age dans le but d'être compatible 
# avec image_dataset_from_directory
# Ajuster le nom du répertoire ou sont décompressés les images de visage et selectionner la plage pour les Ages en fonction
# des performance de votre machine (ajusté pour GPU RTX3080 ici)
data_dir = "C:\\Users\\remyc\\Downloads\\visages\\"  
min_age = 25
max_age = 45
classer_images_par_age(data_dir, min_age, max_age)

train_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,       # Fraction des données utilisée pour la validation
    subset="training",          # Charger les données partie entraînement
    seed=42,                    # Graine pour le découpage des données
    batch_size=32,              # Taille des lots
    image_size=(224, 224),      # redimensionnement des images pour limiter la puissance de calcul nécessaire
)

val_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,       # Fraction des données utilisée pour la validation
    subset="validation",        # Charger les données partie validation
    seed=42,                    # même Graine pour récupérer les 20% restant 
    batch_size=32,              # Taille des lots
    image_size=(224, 224),      # redimensionnement des images pour limiter la puissance de calcul nécessaire
)

In [ ]:
# Nombre de lot dans l'ensemble d'entraînement et verif mémoire puis mise en cache
print("Nombre de batch dans train_ds:", train_ds.cardinality().numpy())  # type: ignore
for x, y in train_ds.take(1):
    taille_batch = x.numpy().nbytes / 1e6
    print(f"Taille d’un batch pour train : {taille_batch:.2f} Mo")
nbatches = len(list(train_ds))  # ou calculé manuellement
total_estime = taille_batch * nbatches
train_class_names = train_ds.class_names
print(f"Taille estimée pour train : {total_estime} Mo")
train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

# Nombre de lot dans l'ensemble de validation
print("Nombre de batch dans val_ds:", val_ds.cardinality().numpy())  # type: ignore
for x, y in val_ds.take(1):
    taille_batch = x.numpy().nbytes / 1e6
    print(f"Taille d’un batch pour train : {taille_batch:.2f} Mo")
nbatches = len(list(val_ds))  # ou calculé manuellement
total_estime = taille_batch * nbatches
print(f"Taille estimée pour test : {total_estime} Mo")
val_class_names = val_ds.class_names
val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
# Affichage aléatoire de 6 images
fig, axs = plt.subplots(2, 3, figsize=(12,8))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for images, labels in train_ds.take(1):
    for j, i in enumerate(np.random.choice(np.arange(0, len(labels)), size=6)):
        img = images[i].numpy().astype("uint8")
        axs[j].axis('off')
        # Affichage de l'image en niveaux de gris
        axs[j].imshow(img)
        # Titre avec le label
        axs[j].set_title(f'Label: {str(labels[i].numpy()+min_age)}')
plt.show()

# 3. Deep learning

>Bonnes pratiques (computer vision)
> - couches de transformation des images (redimensionnement, normalisation et augmentation)
> - couches convolutives de détection des features cachées (filtrage / bord / flou / ...)
> - couches de pooling (réduction de dimension)
> - couche de réduction des connections (pour éviter le surapprentissage)
> - couches denses d'apprentissage

## 3.1 Modèle Tensor Flow Keras (spécialisation image)

#### Creation & Compilation

In [ ]:
# Définition des callback optimisant le temps de traitement
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,                 # critère à observer sur 5 epochs
    min_delta=0.01,             # seuil de détection de critère fixé à 1% de variation
    verbose=1,                  # on log quand on arrête prématurément
    restore_best_weights=True,
    mode='min',                 
)
reduce_lr_on_plateau = ReduceLROnPlateau(
    monitor='val_loss',
    patience=3,                 # critère à observer sur 3 epochs
    min_delta=0.01,             # seuil de détection de critère fixé à 1% de variation
    verbose=1,                  # on ne log que sur l'évènement de réduction
    factor=0.1,                 # facteur de réduction si le cycle est observé
    mode='min',
)
save_on_checkpoint = ModelCheckpoint(
    'nn_tfkeras_vgg16_best.keras',
    save_best_only=True,
    monitor='val_mean_absolute_error',
    mode='min'
)
class TimingCallback(Callback):
    def __init__(self, logs={}):
        self.logs=[]
    def on_epoch_begin(self, epoch, logs={}):
        self.starttime = timer()
    def on_epoch_end(self, epoch, logs={}):
        self.logs.append(timer()-self.starttime)
timing = TimingCallback()

In [ ]:
# Définitions des dimensions d'entrée et de sortie optimisées (nb : pour de la regression linéaire on n'utilisera pas le nombre de classes)
for images, labels in train_ds.take(1):
    shape_image = images.shape[1:]
    break # même si une seule image par sécurité...
print("Shape des images train :", shape_image)  # type: ignore
num_classes = len(train_class_names)
print("Nombre de classes train:", num_classes)

for images, labels in val_ds.take(1):
    shape_image = images.shape[1:]
    break # même si une seule image par sécurité...
print("Shape des images val :", shape_image)  # type: ignore
num_classes = len(val_class_names)
print("Nombre de classes val :", num_classes)

In [ ]:
### Instanciation du modèle par application successive des couches et en réutilisant la structure du modèle vgg16 pre-entrainé
nn_vgg16_core = VGG16(weights='imagenet', include_top=False)
nn_vgg16_core.trainable = False

inputs = Input(shape=shape_image, name="Input")
x = CustomRandomRotation(0.1)(inputs)
x = CustomRandomTranslation(0.1, 0.1)(x)
x = CustomRandomZoom(0.1)(x)
# x = CustomRandomBrightness(0.1)(x) # non recommandé pour vgg16
# x = CustomRandomBrightness(0.1)(x) # non recommandé pour vgg16
# x = CustomRandomContrast(0.1)(x) # non recommandé pour vgg16
# x = Resizing(50,50)(x) # non recommandé pour vgg16
# x = Rescaling(1./255)(x) # non recommandé pour vgg16
x = RandomFlip("horizontal")(x)
x = nn_vgg16_core(x)
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(rate=0.2)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(rate=0.2)(x)
outputs = Dense(1, activation='linear')(x)

nn_tfkeras_vgg16 = Model(inputs=inputs, outputs=outputs)

In [ ]:
# Résumé sur le nombre de paramètres et la structure du modèle
nn_tfkeras_vgg16.summary()

In [ ]:
# Compilation avec les paramètres généraux de comportement (perte/algo/métrique)
nn_tfkeras_vgg16.compile(
    loss='mse',                             # fonction de perte
    optimizer='adam',                       # algorithme d'optimisation
    metrics=['mean_absolute_error'],        # métrique d'évaluation
)

#### Entraînement et métriques

In [ ]:
MODEL_PATH = "nn_tfkeras_vgg16_best.keras"

# Chargement du modèle si le fichier existe
if os.path.exists(MODEL_PATH):
    print(f"🔁 Chargement du modèle depuis {MODEL_PATH} pour continuer l'entrainement")
    nn_tfkeras_vgg16 = load_model(
        MODEL_PATH,
        custom_objects={
            "CustomRandomRotation": CustomRandomRotation,
            "CustomRandomZoom": CustomRandomZoom,
            "CustomRandomBrightness": CustomRandomBrightness,
            "CustomRandomContrast": CustomRandomContrast,
            "CustomRandomTranslation": CustomRandomTranslation,
        }
    )
else:
    print(f"Aucun fichier '{MODEL_PATH}' trouvé. Réentrainement depuis zéro.")

# Prétraitement des images (conforme à VGG16)
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))

# Reprise d'entraînement
training_history = nn_tfkeras_vgg16.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[
        reduce_lr_on_plateau,
        early_stopping,
        save_on_checkpoint,
        timing
    ]
)

In [ ]:
# visualisation des courbe d'apprentissage (MAE / LOSS)
train_mae = training_history.history['mean_absolute_error']
val_mae = training_history.history['val_mean_absolute_error']
train_loss = training_history.history['loss']
val_loss = training_history.history['val_loss']
fig, axs = plt.subplots(1, 2, figsize=(12,6))
axs[0].plot(train_loss, label='Loss (training)')
axs[0].plot(val_loss, label='Loss (validation)')
axs[0].set_title('Loss evolution per epoch')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss')
axs[0].legend()
axs[1].plot(train_mae, label='MAE (training)')
axs[1].plot(val_mae, label='MAE (validation)')
axs[1].set_title('MAE evolution per epoch')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('MAE')
axs[1].legend()
plt.show()

#### Prédiction et visualisation

In [ ]:
# Prédictions du modèle : pour chaque échantillon, une regression linéaire sur l'age relatif predit
# nous commencons avec l'échelle (0 <=> min_age) qu'il faudra restituer après avoir arrondi l'age a l'entier le plus proche
y_test_prob = nn_tfkeras_vgg16.predict(val_ds)
y_test_pred_class = np.rint(y_test_prob).astype(int).flatten()
y_test_pred_age = y_test_pred_class + min_age

y_test = []
for _, labels in val_ds.unbatch():
    y_test.append(labels.numpy())
y_test = np.array(y_test, dtype=int)
y_test_age = y_test + min_age

In [ ]:
# Étape 1 — Liste des erreurs grossières
error_indexes = []
for i in range(len(y_test_pred_age)):
    if (np.abs(y_test_pred_age[i] - y_test_age[i])>8):
        error_indexes += [i]
print(f"Nombre d'erreur grossières du modèle [{len(error_indexes)}]")

# Étape 2 — Récupération des images associées dans l'ensemble de validation
val_ds_reloaded = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,       # Fraction des données utilisée pour la validation
    subset="validation",        # Charger les données partie validation
    seed=42,                    # même Graine pour récupérer les 20% restant 
    batch_size=32,              # Taille des lots
    image_size=(224, 224),      # redimensionnement des images pour limiter la puissance de calcul nécessaire
)
images_list = []
for batch_images, _ in val_ds_reloaded.unbatch().batch(1).take(len(y_test_pred_age)):
    images_list.append(batch_images[0].numpy())  # batch_images est de forme (1, H, W, C)
images_array = np.array(images_list)  # (N, H, W, C)

# Étape 3 — Affichage aléatoire des images sur lesquelles le modèle s'est trompé
(max_line, max_col) = (3, 3)
print(f"On en affiche aléatoirement {max_line}x{max_col} images mal classées")
fig, axs = plt.subplots(max_line, max_col, figsize=(max_line*4,max_col*4))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for j, i in enumerate(np.random.choice(error_indexes, size=max_line*max_col, replace=False)):
    axs[j].axis('off')
    # Affichage de l'image
    axs[j].imshow(images_array[i].astype("uint8"))
    # Titre avec le label
    axs[j].set_title(
        f'True Label: {str(y_test_age[i])} | '
        f'Prediction: {str(y_test_pred_age[i])}'
    )

# 4. Interprétabilité

#### Chargement d'une image unitaire

In [ ]:
def load_and_preprocess_images(paths, target_size=(224, 224)):
    """Charge et prétraite une liste d’images depuis leurs chemins."""
    imgs = []
    for path in paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Image non trouvée : {path}")
        img = image.load_img(path, target_size=target_size)
        img_array = image.img_to_array(img)
        imgs.append(img_array)
    imgs = np.array(imgs)
    return preprocess_input(imgs)  # adapté à VGG16

# === PARAMÈTRES ===
img_paths = [
    "C:\\Users\\remyc\\Downloads\\visage_perso\\25_remy.jpg",
    "C:\\Users\\remyc\\Downloads\\visage_perso\\44_remy.jpg"
]
model = nn_tfkeras_vgg16
img_size = (224, 224)

# === PRÉTRAITEMENT ===
img_batch_pp = load_and_preprocess_images(img_paths, img_size)
print(img_batch_pp.shape)

# === PRÉDICTION ===
y_pred_scaled = model.predict(img_batch_pp)          # valeur flottante entre 0 et (max_age - min_age)
y_pred_ages = y_pred_scaled.flatten() + min_age
y_pred_ages = np.round(y_pred_ages, 2)

# === AFFICHAGE ===
for i, age in enumerate(y_pred_ages):
    print(f"Image {i+1} : âge prédit = {age:.2f} ans")

In [ ]:
def find_layer_by_name(model, layer_name):
    for layer in model.layers:
        if layer.name == layer_name:
            return layer
        # Si la couche elle-même est un sous-modèle, recherche récursive
        if isinstance(layer, Model):
            found = find_layer_by_name(layer, layer_name)
            if found is not None:
                return found
    return None  # Si non trouvée

def forward_to_layer(image, model, layer_name):
    """
    Rejoue manuellement le forward pass jusqu'à la couche 'layer_name'
    (même si imbriquée dans un sous-modèle).
    Retourne les activations de cette couche.
    """
    x = image
    if x.ndim == 3:
        x = tf.expand_dims(x, axis=0)

    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            for sub_layer in layer.layers:
                x = sub_layer(x)
                if sub_layer.name == layer_name:
                    return x
        else:
            x = layer(x)
            if layer.name == layer_name:
                return x
    raise ValueError(f"Couche '{layer_name}' non trouvée dans le modèle.")

def forward_from_layer(input_tensor, model, start_layer_name):
    """Rejoue le modèle à partir de `start_layer_name` jusqu’à la sortie"""
    x = input_tensor
    past_target = False
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            for sub_layer in layer.layers:
                if past_target:
                    x = sub_layer(x)
                elif sub_layer.name == start_layer_name:
                    past_target = True
        else:
            if past_target:
                x = layer(x)
            elif layer.name == start_layer_name:
                past_target = True
    return x

def get_all_conv2d_layers(model):
    conv_layers = []
    for layer in model.layers:
        if isinstance(layer, Conv2D):
            conv_layers.append(layer.name)
        elif isinstance(layer, Model):
            # Récursion sur les sous-modèles imbriqués (comme VGG16)
            conv_layers += get_all_conv2d_layers(layer)
    return conv_layers

#### Grad Cam

In [ ]:
def grad_cam(image, model, layer_name):
    # Trouver la couche cible
    target_layer = find_layer_by_name(model, layer_name)
    if target_layer is None:
        raise ValueError(f"Couche '{layer_name}' non trouvée dans le modèle.")
    elif target_layer==layer_name:
        print(f"Couche '{layer_name}' trouvée dans le modèle.")
    else:
        print(f"Couche '{layer_name}' trouvée dans un sous modèle.")
    
    # Ajout d'une dimension de batch
    if image.ndim == 3:
        image = tf.expand_dims(image, axis=0)

    # Calcul du forward vers le layer donné + enregistrement dans GradientTape
    with tf.GradientTape() as tape:
        conv_outputs = forward_to_layer(image, model, layer_name)
        tape.watch(conv_outputs)
        predictions = forward_from_layer(conv_outputs, model, layer_name)
        # Régression (1 sortie) ou classification (plusieurs classes)
        if predictions.shape[-1] == 1:
            loss = predictions[:, 0]
        else:
            predicted_class = tf.argmax(predictions[0])
            loss = predictions[:, predicted_class]

    print(f"{layer_name} conv_outputs shape:", conv_outputs.shape)
    print(f"{layer_name} prediction shape:", predictions.shape)
    # Gradients des scores par rapport aux sorties de la couche convolutive
    grads = tape.gradient(loss, conv_outputs)

    # Moyenne pondérée des gradients pour chaque canal
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Pondération des activations par les gradients calculés
    conv_outputs = conv_outputs[0]  # Supprimer la dimension batch
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    # Normalisation de la carte de chaleur
    heatmap = tf.maximum(heatmap, 0)  # Se concentrer uniquement sur les valeurs positives
    max_val = tf.reduce_max(heatmap)
    heatmap = heatmap / (max_val + tf.keras.backend.epsilon())  # normaliser en évitant la division par 0
    heatmap = heatmap.numpy()  # Convertir en tableau numpy pour la visualisation

   # Redimensionner la carte de chaleur pour correspondre à la taille de l'image d'origine
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (image.shape[1], image.shape[2])).numpy()
    heatmap_resized = np.squeeze(heatmap_resized, axis=-1) # supprimer la dimension de taille 1 à la fin du tableau heatmap_resized

    # Colorier la carte de chaleur avec une palette (par exemple, "jet")
    heatmap_colored = plt.cm.jet(heatmap_resized)[..., :3] # Récupérer les canaux R, G, B 

    # Superposition sur l'image d'entrée (normalisée entre 0-1)
    original_img = image[0].numpy()
    if original_img.max() > 1.0:
        original_img = original_img / 255.0

    superimposed_image = np.clip(heatmap_colored * 0.7 + original_img, 0, 1)

    return superimposed_image, float(predictions[0][0])

In [ ]:
def show_grad_cam_cnn(images, model, layers_to_plot=None):
    if images.ndim == 3:
        images = np.expand_dims(images, axis=0)

    number_of_images = images.shape[0]
    conv_layers = layers_to_plot or get_all_conv2d_layers(model)
    print(f"Layers sélectionnés : {conv_layers}")

    max_line = len(conv_layers)
    max_col = number_of_images

    fig, axs = plt.subplots(max_line, max_col, figsize=(max_col * 4, max_line * 4))
    # Force axs en tableau 2D (cas 1x1, 1xN ou Nx1) pour gérer un display unifié axs[i, j] même avec 1 ligne ou 1 colonne
    if max_line == 1 and max_col == 1:
        axs = np.array([[axs]])
    elif max_line == 1:
        axs = np.expand_dims(axs, axis=0)  # shape (1, N)
    elif max_col == 1:
        axs = np.expand_dims(axs, axis=1)  # shape (N, 1)
    for j, layer in enumerate(conv_layers): 
        for i in range(number_of_images):
            grad_cam_image, pred = grad_cam(images[i], model, layer)
            axs[j, i].imshow(grad_cam_image)
            axs[j, i].set_title(f'{layer}\nÂge prédit : {pred+min_age:.2f}')
            axs[j, i].axis('off')
    plt.show()

show_grad_cam_cnn(img_batch_pp, model, ['block5_conv1', 'block5_conv2', 'block5_conv3'])

#### SHAP

In [ ]:
def compute_and_plot_shap(images, model, max_evals=500, top_classes=4, verbose=True):
    """
    Calcule et affiche les valeurs SHAP pour un modèle Keras/TensorFlow
    en classification ou régression à partir d'un batch d'images.

    Args:
        model: modèle Keras entraîné
        images: tableau numpy (batch d’images), shape (N, H, W, C)
        max_evals: nombre max d'évaluations SHAP
        top_classes: nombre de classes à afficher pour classification
        verbose: affiche les formes des tenseurs si True
    """
    # Étape 1 : Masker
    masker = shap.maskers.Image("inpaint_telea", images[0].shape)

    # Étape 2 : Explainer
    explainer = shap.Explainer(model, masker)

    # Étape 3 : inférence pour connaître la forme de sortie
    predictions = model.predict(images[:1])
    output_dim = predictions.shape[-1]

    if verbose:
        print(f"Predictions shape: {predictions.shape}")
        print("Mode :", "régression" if output_dim == 1 else "classification")

    # Étape 4 : Calcul des SHAP values
    if output_dim == 1:
        shap_values = explainer(images, max_evals=max_evals)
    else:
        shap_values = explainer(
            images,
            max_evals=max_evals,
            outputs=shap.Explanation.argsort.flip[:top_classes]
        )

    # Étape 5 : Affichage
    shap.image_plot(shap_values)

In [ ]:
compute_and_plot_shap(img_batch_pp, model, max_evals=500)

#### Feature map

In [ ]:
def show_feature_maps(image, model, layers_to_plot=None, max_feats_per_layer=16):
    """
    Affiche les feature maps de certaines couches Conv2D d'un modèle (y compris imbriquées).
    
    Args:
        image: tableau numpy (H, W, C)
        model: modèle Keras avec couches convolutives
        layers_to_plot: liste optionnelle de noms de couches à afficher
        max_feats_per_layer: nombre maximal de filtres à afficher par couche
    """
    conv_layers = layers_to_plot or get_all_conv2d_layers(model)

    for layer_name in conv_layers:
        try:
            feature_maps = forward_to_layer(image, model, layer_name).numpy()
        except Exception as e:
            print(f"⚠️ Impossible d'obtenir les activations pour {layer_name} : {e}")
            continue

        feature_maps = np.squeeze(feature_maps)
        if feature_maps.ndim != 3:
            continue  # Ne rien afficher si la forme n'est pas attendue (H, W, N)

        h, w, n_filters = feature_maps.shape
        n_to_display = min(n_filters, max_feats_per_layer)
        grid_size = int(np.ceil(np.sqrt(n_to_display)))

        plt.figure(figsize=(grid_size * 2, grid_size * 2))
        for i in range(n_to_display):
            plt.subplot(grid_size, grid_size, i + 1)
            plt.imshow(feature_maps[..., i], cmap='viridis')
            plt.axis('off')
            plt.title(f'Filtre {i + 1}', fontsize=8)
        plt.suptitle(f'Activations : {layer_name} [{n_to_display}/{n_filters}]', fontsize=12)
        plt.tight_layout()
        plt.show()

In [ ]:
show_feature_maps(img_batch_pp[0], model, ['block5_conv2'])